# TRIBE v2 Thumbnail Study: Brain Responses vs YouTube Views

This notebook adapts the main TRIBE v2 Colab flow to a thumbnail-comparison experiment.

It will:
1. Load the pretrained TRIBE v2 model in a video-only inference mode
2. Pull recent uploads from a YouTube channel that **you specify**
3. Keep uploads that are at least 14 days old and long enough to behave like normal thumbnail-driven videos
4. Select a mix of high-view and low-view videos
5. Download their thumbnails, turn them into short silent clips, and run inference
6. Compare brain-response summary metrics against current YouTube view counts

Important caveats:
- This is an exploratory visual-salience study, not a validated predictor of thumbnail performance.
- A static thumbnail repeated as a short silent clip is out-of-distribution relative to natural video.
- Correlations with `n=10` should be treated as descriptive, not causal.


## Setup

For Colab:
1. Enable a GPU runtime
2. Run the install cell below
3. **Restart the runtime manually after the install completes**
4. Re-run the notebook from the top

Why this notebook is strict about setup:
- some TRIBE v2 dependencies can break when NumPy 2.1+ is left in the runtime
- changing NumPy/SciPy in-place without a restart can leave the kernel in a mixed state
- the install cell below finishes by stopping execution on purpose so you do not continue on a stale kernel


In [ ]:
from importlib import metadata
from importlib import import_module
import subprocess
import sys


def parse_major_minor(version: str) -> tuple[int, int]:
    parts = []
    for token in version.split("."):
        digits = "".join(ch for ch in token if ch.isdigit())
        if not digits:
            break
        parts.append(int(digits))
        if len(parts) == 2:
            break
    while len(parts) < 2:
        parts.append(0)
    return tuple(parts)


required_dists = [
    "tribev2",
    "yt-dlp",
    "Pillow",
    "exca",
    "nibabel",
    "nilearn",
    "neuralset",
    "neuraltrain",
]
missing = []
installed = {}
for dist_name in required_dists + ["numpy", "scipy"]:
    try:
        installed[dist_name] = metadata.version(dist_name)
    except metadata.PackageNotFoundError:
        missing.append(dist_name)

numpy_ok = False
scipy_ok = False
if "numpy" in installed:
    numpy_ok = parse_major_minor(installed["numpy"]) < (2, 1)
if "scipy" in installed:
    scipy_mm = parse_major_minor(installed["scipy"])
    scipy_ok = (1, 13) <= scipy_mm < (1, 16)

import_failures = []
for module_name in ["PIL", "moviepy", "exca", "nibabel", "nilearn", "neuralset", "neuraltrain"]:
    try:
        import_module(module_name)
    except Exception as exc:
        import_failures.append(f"{module_name}: {type(exc).__name__}: {exc}")

needs_install = bool(missing) or not numpy_ok or not scipy_ok or bool(import_failures)

if needs_install:
    print("Installing notebook dependencies...")
    if missing:
        print("Missing distributions:", ", ".join(missing))
    if import_failures:
        print("Broken imports detected:")
        for failure in import_failures:
            print(" -", failure)
    if "numpy" in installed:
        print("Detected NumPy:", installed["numpy"])
    if "scipy" in installed:
        print("Detected SciPy:", installed["scipy"])

    # Install the full notebook stack first, then force NumPy/SciPy back to a
    # known-good range for TRIBE's compiled dependencies.
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--upgrade",
            "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git",
            "exca",
            "yt-dlp",
            "pillow",
            "pandas",
            "matplotlib",
            "moviepy",
        ],
        check=True,
    )

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--force-reinstall",
            "numpy>=1.26.4,<2.1",
            "scipy>=1.13,<1.16",
        ],
        check=True,
    )

    raise SystemExit(
        "Install complete. Restart the runtime now, then run the notebook again from the top."
    )

print("Setup already looks correct.")
print(f"NumPy version: {installed['numpy']}")
print(f"SciPy version: {installed['scipy']}")


### Channel handle

Edit the next cell and set `CHANNEL_HANDLE_INPUT`.

Handle format:
- you can enter `channelname`
- you can enter `@channelname`
- the notebook will normalize it for you
- do **not** paste the full channel URL


In [ ]:
# User config
# Enter the channel handle only. Both "channelname" and "@channelname" work.
CHANNEL_HANDLE_INPUT = "@channelname"

CHANNEL_HANDLE = CHANNEL_HANDLE_INPUT.strip()
if not CHANNEL_HANDLE:
    raise ValueError("Please set CHANNEL_HANDLE_INPUT to a YouTube handle such as 'channelname' or '@channelname'.")
if CHANNEL_HANDLE.startswith("http://") or CHANNEL_HANDLE.startswith("https://"):
    raise ValueError("Set CHANNEL_HANDLE_INPUT to the handle only, not the full URL.")
if not CHANNEL_HANDLE.startswith("@"):
    CHANNEL_HANDLE = f"@{CHANNEL_HANDLE}"
CHANNEL_URL = f"https://www.youtube.com/{CHANNEL_HANDLE}/videos"


## Load TRIBE v2

After setting the channel handle above, run the imports/runtime cell and then the model-loading cell.

The released checkpoint was trained with text, audio, and video features, but for this thumbnail study we intentionally run **video-only inference**. That avoids the gated Llama dependency and keeps the experiment focused on visual responses.

The imports cell also checks that the restarted runtime is actually using the expected NumPy/SciPy versions before loading TRIBE.


In [ ]:
from pathlib import Path
import json
import math
import os
import re
import subprocess
import textwrap

MPLCONFIG_DIR = Path("./cache/matplotlib")
MPLCONFIG_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIG_DIR.resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
try:
    from moviepy import ImageClip
except ImportError:
    from moviepy.editor import ImageClip

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

try:
    import scipy
except ImportError as exc:
    msg = str(exc)
    if "numpy._core.umath" in msg or "_center" in msg:
        raise RuntimeError(
            "SciPy imported against an incompatible NumPy runtime. "
            "Run the setup cell, restart the runtime, and rerun the notebook from the top."
        ) from exc
    raise


def parse_major_minor(version: str) -> tuple[int, int]:
    parts = []
    for token in version.split("."):
        digits = "".join(ch for ch in token if ch.isdigit())
        if not digits:
            break
        parts.append(int(digits))
        if len(parts) == 2:
            break
    while len(parts) < 2:
        parts.append(0)
    return tuple(parts)


numpy_mm = parse_major_minor(np.__version__)
scipy_mm = parse_major_minor(scipy.__version__)
print(f"NumPy version: {np.__version__}")
print(f"SciPy version: {scipy.__version__}")

if numpy_mm >= (2, 1):
    raise RuntimeError(
        "This notebook expects NumPy < 2.1 for TRIBE compatibility. "
        "Run the setup cell, restart the runtime, and then rerun the notebook from the top."
    )

if not ((1, 13) <= scipy_mm < (1, 16)):
    raise RuntimeError(
        "This notebook expects SciPy >= 1.13 and < 1.16. "
        "Run the setup cell, restart the runtime, and then rerun the notebook from the top."
    )

try:
    from tribev2.demo_utils import TribeModel, download_file
    from tribev2.plotting.cortical import PlotBrainNilearn
    from tribev2.utils import get_hcp_labels, summarize_by_roi
except ImportError as exc:
    msg = str(exc)
    if "numpy._core.umath" in msg or "_center" in msg:
        raise RuntimeError(
            "TRIBE imported against an incompatible NumPy/SciPy runtime. "
            "Run the setup cell, restart the runtime, and rerun the notebook from the top."
        ) from exc
    if isinstance(exc, ModuleNotFoundError) or any(
        missing_mod in msg for missing_mod in ["exca", "nibabel", "nilearn", "neuralset", "neuraltrain"]
    ):
        raise RuntimeError(
            "Notebook dependencies are incomplete. Run the setup cell, restart the runtime, and rerun the notebook from the top."
        ) from exc
    raise

CACHE_FOLDER = Path("./cache")
STUDY_FOLDER = Path("./thumbnail_brain_study")
THUMB_FOLDER = STUDY_FOLDER / "thumbnails"
CLIP_FOLDER = STUDY_FOLDER / "clips"
OUT_FOLDER = STUDY_FOLDER / "outputs"
for path in [CACHE_FOLDER, STUDY_FOLDER, THUMB_FOLDER, CLIP_FOLDER, OUT_FOLDER]:
    path.mkdir(parents=True, exist_ok=True)



In [ ]:
DAYS_OLD_MIN = 14
TOP_N = 5
BOTTOM_N = 5
MIN_DURATION_SEC = 180
THUMBNAIL_VIDEO_SECONDS = 6
THUMBNAIL_FPS = 10

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
    config_update={"data": {"features_to_use": ["video"]}},
)
plotter = PlotBrainNilearn(mesh="fsaverage5")


## Select 10 videos from the chosen channel

Selection rule:
- use the selected channel's video list as the candidate pool
- keep videos at least 14 days old
- keep long-form uploads (`duration >= 180s`) so the thumbnail matters in the usual YouTube way
- choose the 5 highest-view and 5 lowest-view eligible videos based on **current** view counts at notebook runtime


In [ ]:
def run_yt_dlp_json(url: str) -> dict:
    cmd = [
        "yt-dlp",
        "--skip-download",
        "--dump-single-json",
        "--no-warnings",
        url,
    ]
    payload = subprocess.check_output(cmd, text=True)
    return json.loads(payload)


def list_channel_video_urls(channel_url: str, playlist_end: int = 80) -> list[str]:
    cmd = [
        "yt-dlp",
        "--flat-playlist",
        "--dump-single-json",
        "--playlist-end",
        str(playlist_end),
        "--no-warnings",
        channel_url,
    ]
    payload = subprocess.check_output(cmd, text=True)
    info = json.loads(payload)
    entries = info.get("entries") or []
    urls = []
    for entry in entries:
        video_id = entry.get("id")
        if video_id:
            urls.append(f"https://www.youtube.com/watch?v={video_id}")
    if not urls:
        raise RuntimeError(f"Could not list videos from {channel_url}")
    return urls


def parse_upload_date(upload_date: str):
    parsed = pd.to_datetime(upload_date, format="%Y%m%d", utc=True, errors="coerce")
    if pd.isna(parsed):
        parsed = pd.to_datetime(upload_date, utc=True, errors="coerce")
    return parsed


cutoff = pd.Timestamp.now(tz="UTC") - pd.Timedelta(days=DAYS_OLD_MIN)
video_urls = list_channel_video_urls(CHANNEL_URL, playlist_end=80)

candidate_rows = []
for video_url in video_urls:
    info = run_yt_dlp_json(video_url)
    upload_date = info.get("upload_date")
    if not upload_date:
        continue
    published = parse_upload_date(upload_date)
    if pd.isna(published):
        continue
    if published > cutoff:
        continue

    duration = info.get("duration")
    view_count = info.get("view_count")
    if duration is None or duration < MIN_DURATION_SEC:
        continue
    if view_count is None or int(view_count) <= 0:
        continue

    candidate_rows.append(
        {
            "video_id": info["id"],
            "title": info.get("title", info["id"]),
            "upload_date": pd.Timestamp(info["upload_date"]),
            "published_at": published.tz_convert(None),
            "view_count": int(view_count),
            "duration_sec": int(duration),
            "thumbnail": info.get("thumbnail") or f"https://i.ytimg.com/vi/{info['id']}/hqdefault.jpg",
            "webpage_url": info.get("webpage_url", video_url),
        }
    )

if not candidate_rows:
    raise RuntimeError(
        "No eligible videos were found. Check the channel handle, or reduce DAYS_OLD_MIN / MIN_DURATION_SEC."
    )

candidate_df = pd.DataFrame(candidate_rows).sort_values("view_count", ascending=False)
if len(candidate_df) < TOP_N + BOTTOM_N:
    raise RuntimeError(
        f"Only found {len(candidate_df)} eligible videos. Try reducing MIN_DURATION_SEC or increasing the feed window."
    )

top_df = candidate_df.nlargest(TOP_N, "view_count").assign(selection_bucket="top_views")
bottom_df = candidate_df.nsmallest(BOTTOM_N, "view_count").assign(selection_bucket="low_views")
selected_df = pd.concat([top_df, bottom_df], ignore_index=True).drop_duplicates("video_id")

if len(selected_df) < TOP_N + BOTTOM_N:
    needed = TOP_N + BOTTOM_N - len(selected_df)
    filler = candidate_df.loc[~candidate_df.video_id.isin(selected_df.video_id)].head(needed).assign(selection_bucket="fill")
    selected_df = pd.concat([selected_df, filler], ignore_index=True)

selected_df = selected_df.sort_values(["selection_bucket", "view_count"], ascending=[True, False]).reset_index(drop=True)
selected_df["views_millions"] = selected_df["view_count"] / 1_000_000

display(
    selected_df[
        [
            "selection_bucket",
            "title",
            "published_at",
            "duration_sec",
            "view_count",
            "webpage_url",
        ]
    ]
)


## Download thumbnails and convert them to short video clips

We create silent 6-second clips by repeating each thumbnail at a fixed frame rate. This is intentionally simple: the goal is a controlled visual comparison, not a natural-video simulation.


In [ ]:
def safe_slug(text: str, max_len: int = 80) -> str:
    text = re.sub(r"[^a-zA-Z0-9]+", "-", text).strip("-").lower()
    return text[:max_len] or "item"


def ensure_thumbnail_clip(row: pd.Series):
    slug = safe_slug(row["title"])
    thumb_path = THUMB_FOLDER / f"{row['video_id']}-{slug}.jpg"
    clip_path = CLIP_FOLDER / f"{row['video_id']}-{slug}-{THUMBNAIL_VIDEO_SECONDS}s.mp4"

    if not thumb_path.exists():
        download_file(row["thumbnail"], thumb_path)

    if not clip_path.exists():
        clip = ImageClip(str(thumb_path), duration=THUMBNAIL_VIDEO_SECONDS)
        clip.write_videofile(
            str(clip_path),
            fps=THUMBNAIL_FPS,
            codec="libx264",
            audio=False,
            logger=None,
        )
        clip.close()

    return thumb_path, clip_path


materialized_rows = []
for _, row in selected_df.iterrows():
    thumb_path, clip_path = ensure_thumbnail_clip(row)
    item = row.to_dict()
    item["thumbnail_path"] = str(thumb_path)
    item["clip_path"] = str(clip_path)
    materialized_rows.append(item)

study_df = pd.DataFrame(materialized_rows)
study_df.to_csv(OUT_FOLDER / "selected-thumbnail-videos.csv", index=False)

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
flat_axes = list(axes.flat)
for ax, row in zip(flat_axes, study_df.itertuples()):
    ax.imshow(Image.open(row.thumbnail_path))
    ax.axis("off")
    ax.set_title(
        textwrap.fill(row.title, 24)
        + f"\n{row.view_count:,} views\n[{row.selection_bucket}]",
        fontsize=8,
    )

for ax in flat_axes[len(study_df):]:
    ax.axis("off")

plt.suptitle("Selected thumbnails", fontsize=14)
plt.tight_layout()
thumbnail_gallery_path = OUT_FOLDER / "selected-thumbnails.png"
plt.savefig(thumbnail_gallery_path, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved thumbnail gallery to: {thumbnail_gallery_path}")


## Run inference on each thumbnail clip

We bypass the notebook's `get_events_dataframe(video_path=...)` convenience path and build direct `Video` events instead. That keeps the experiment visual-only and avoids audio/transcript steps.


In [ ]:
left_labels = list(get_hcp_labels(mesh="fsaverage5", combine=False, hemi="left").keys())
right_labels = list(get_hcp_labels(mesh="fsaverage5", combine=False, hemi="right").keys())
roi_labels = [f"{name}-lh" for name in left_labels] + [f"{name}-rh" for name in right_labels]

roi_groups = {
    "visual": {
        "V1", "V2", "V3", "V4", "V3A", "V3B", "V6", "V6A", "V7",
        "LO1", "LO2", "LO3", "MT", "MST", "FFC", "PIT", "PHA1", "PHA2", "PHA3",
        "VMV1", "VMV2", "VMV3", "VVC",
    },
    "auditory_speech": {
        "A1", "A4", "A5", "LBelt", "MBelt", "PBelt", "RI", "STGa", "TA2",
        "STSda", "STSdp", "STSva", "STSvp",
    },
    "language_semantic": {
        "44", "45", "47l", "p47r", "IFJa", "IFJp", "IFSa", "IFSp",
        "TGd", "TGv", "TE1a", "TE1p", "TE2a", "TE2p",
    },
    "attention_control": {
        "FEF", "PEF", "IP0", "IP1", "IP2", "LIPd", "LIPv", "AIP", "MIP", "VIP",
        "8Av", "8Ad", "8BL", "8C", "9-46d", "9-46v", "SCEF",
    },
    "motor_somato": {
        "1", "2", "3a", "3b", "4", "5m", "5mv", "5L", "5hv",
        "6a", "6d", "6ma", "6mp", "6r", "6v", "24dd", "24dv",
    },
}


def strip_hemi(roi_name: str) -> str:
    return roi_name.replace("-lh", "").replace("-rh", "")


group_indices = {
    group_name: [i for i, roi in enumerate(roi_labels) if strip_hemi(roi) in members]
    for group_name, members in roi_groups.items()
}


def build_video_events(video_path: str, duration_sec: float) -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "type": "Video",
                "filepath": str(video_path),
                "start": 0.0,
                "duration": float(duration_sec),
                "timeline": Path(video_path).stem,
                "subject": "thumbnail-study",
            }
        ]
    )


prediction_cache = {}
clip_rows = []
roi_time_rows = []

for row in study_df.itertuples():
    events = build_video_events(row.clip_path, THUMBNAIL_VIDEO_SECONDS)
    preds, segments = model.predict(events=events, verbose=True)
    prediction_cache[row.video_id] = {"preds": preds, "segments": segments, "meta": row._asdict()}

    top_scores = []
    group_series = {name: [] for name in roi_groups}
    peak_vertex_by_t = []
    top_rois = []

    for t in range(preds.shape[0]):
        roi_scores = summarize_by_roi(preds[t], hemi="both_separate", mesh="fsaverage5")
        top_idx = int(np.argmax(roi_scores))
        top_score = float(roi_scores[top_idx])
        top_roi = roi_labels[top_idx]
        top_scores.append(top_score)
        top_rois.append(top_roi)
        peak_vertex_by_t.append(float(np.max(preds[t])))

        per_group = {}
        for group_name, indices in group_indices.items():
            group_value = float(np.mean(roi_scores[indices])) if indices else np.nan
            group_series[group_name].append(group_value)
            per_group[group_name] = group_value

        roi_time_rows.append(
            {
                "video_id": row.video_id,
                "title": row.title,
                "view_count": row.view_count,
                "selection_bucket": row.selection_bucket,
                "timestep": t,
                "top_roi": top_roi,
                "top_roi_score": top_score,
                "peak_vertex_response": float(np.max(preds[t])),
                **per_group,
            }
        )

    peak_timestep = int(np.argmax(top_scores))
    clip_rows.append(
        {
            "video_id": row.video_id,
            "title": row.title,
            "upload_date": pd.Timestamp(row.upload_date).date().isoformat(),
            "view_count": int(row.view_count),
            "selection_bucket": row.selection_bucket,
            "webpage_url": row.webpage_url,
            "thumbnail_path": row.thumbnail_path,
            "clip_path": row.clip_path,
            "n_timesteps": int(preds.shape[0]),
            "peak_timestep": peak_timestep,
            "peak_top_roi": top_rois[peak_timestep],
            "peak_top_roi_score": float(np.max(top_scores)),
            "mean_top_roi_score": float(np.mean(top_scores)),
            "peak_vertex_response": float(np.max(preds)),
            "mean_vertex_response": float(np.mean(preds)),
            "visual_mean": float(np.mean(group_series["visual"])),
            "auditory_speech_mean": float(np.mean(group_series["auditory_speech"])),
            "language_semantic_mean": float(np.mean(group_series["language_semantic"])),
            "attention_control_mean": float(np.mean(group_series["attention_control"])),
            "motor_somato_mean": float(np.mean(group_series["motor_somato"])),
            "visual_peak": float(np.max(group_series["visual"])),
            "attention_control_peak": float(np.max(group_series["attention_control"])),
        }
    )

results_df = pd.DataFrame(clip_rows).sort_values("view_count", ascending=False).reset_index(drop=True)
roi_time_df = pd.DataFrame(roi_time_rows)

results_path = OUT_FOLDER / "thumbnail-study-results.csv"
roi_time_path = OUT_FOLDER / "thumbnail-study-roi-timeseries.csv"
results_df.to_csv(results_path, index=False)
roi_time_df.to_csv(roi_time_path, index=False)

display(results_df)
print(f"Saved: {results_path}")
print(f"Saved: {roi_time_path}")


## Visualize the brain response for each thumbnail

We export two views:
- a compact gallery with the thumbnail and its **peak-timestep** cortical response map
- a full per-thumbnail timestep gallery so you can inspect the brain response at **every** timestep, not just the peak


In [ ]:
import shutil

fig = plt.figure(figsize=(20, 17.5))

for i, row in enumerate(results_df.itertuples(), start=1):
    thumb_ax = fig.add_subplot(5, 4, 2 * i - 1)
    thumb_ax.imshow(Image.open(row.thumbnail_path))
    thumb_ax.axis("off")
    thumb_ax.set_title(
        textwrap.fill(row.title, 28)
        + f"\n{row.view_count:,} views\npeak ROI: {row.peak_top_roi}",
        fontsize=8,
    )

    brain_ax = fig.add_subplot(5, 4, 2 * i, projection="3d")
    peak_brain = prediction_cache[row.video_id]["preds"][row.peak_timestep]
    plotter.plot_surf(
        peak_brain,
        axes=brain_ax,
        views="left",
        cmap="fire",
        norm_percentile=99,
        vmin=0.6,
        alpha_cmap=(0, 0.2),
    )
    brain_ax.set_title(
        f"peak t={row.peak_timestep}\nROI={row.peak_top_roi}\npeak score={row.peak_top_roi_score:.3f}",
        fontsize=8,
    )

plt.suptitle("Thumbnail study: thumbnail vs peak ROI response", fontsize=16)
plt.subplots_adjust(left=0.03, right=0.99, top=0.95, bottom=0.05, wspace=0.12, hspace=0.32)
brain_gallery_path = OUT_FOLDER / "thumbnail-study-peak-brain-gallery.png"
plt.savefig(brain_gallery_path, dpi=180, bbox_inches="tight", pad_inches=0.25)
plt.show()

TIMESTEP_GALLERY_DIR = OUT_FOLDER / "thumbnail-study-timestep-galleries"
TIMESTEP_GALLERY_DIR.mkdir(parents=True, exist_ok=True)
timestep_manifest_rows = []

for row in results_df.itertuples():
    preds = prediction_cache[row.video_id]["preds"]
    time_rows = (
        roi_time_df.loc[roi_time_df.video_id == row.video_id]
        .sort_values("timestep")
        .reset_index(drop=True)
    )
    n_timesteps = preds.shape[0]
    fig = plt.figure(figsize=(2.8 * (n_timesteps + 1), 4.2))

    thumb_ax = fig.add_subplot(1, n_timesteps + 1, 1)
    thumb_ax.imshow(Image.open(row.thumbnail_path))
    thumb_ax.axis("off")
    thumb_ax.set_title(
        textwrap.fill(row.title, 24) + f"\n{row.view_count:,} views",
        fontsize=8,
    )

    for t in range(n_timesteps):
        ax = fig.add_subplot(1, n_timesteps + 1, t + 2, projection="3d")
        plotter.plot_surf(
            preds[t],
            axes=ax,
            views="left",
            cmap="fire",
            norm_percentile=99,
            vmin=0.6,
            alpha_cmap=(0, 0.2),
        )
        trow = time_rows.iloc[t]
        ax.set_title(
            f"t={t}\n{trow.top_roi}\n{trow.top_roi_score:.3f}",
            fontsize=7,
        )

    fig.suptitle(f"{row.video_id}: thumbnail + all timesteps", fontsize=12)
    plt.subplots_adjust(left=0.02, right=0.99, top=0.84, bottom=0.10, wspace=0.08)
    gallery_path = TIMESTEP_GALLERY_DIR / f"{row.video_id}-{safe_slug(row.title, max_len=48)}-timesteps.png"
    fig.savefig(gallery_path, dpi=180, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)
    timestep_manifest_rows.append(
        {
            "video_id": row.video_id,
            "title": row.title,
            "view_count": int(row.view_count),
            "timesteps": int(n_timesteps),
            "gallery_path": str(gallery_path),
        }
    )

timestep_manifest_df = pd.DataFrame(timestep_manifest_rows)
timestep_manifest_path = OUT_FOLDER / "thumbnail-study-timestep-galleries-manifest.csv"
timestep_manifest_df.to_csv(timestep_manifest_path, index=False)
timestep_zip_base = OUT_FOLDER / "thumbnail-study-timestep-galleries"
timestep_zip_path = shutil.make_archive(str(timestep_zip_base), "zip", root_dir=TIMESTEP_GALLERY_DIR)

max_timesteps = max(prediction_cache[row.video_id]["preds"].shape[0] for row in results_df.itertuples())
master_fig = plt.figure(figsize=(2.4 * (max_timesteps + 1), 3.25 * len(results_df)))

for row_idx, row in enumerate(results_df.itertuples(), start=0):
    preds = prediction_cache[row.video_id]["preds"]
    time_rows = (
        roi_time_df.loc[roi_time_df.video_id == row.video_id]
        .sort_values("timestep")
        .reset_index(drop=True)
    )
    n_timesteps = preds.shape[0]

    thumb_ax = master_fig.add_subplot(len(results_df), max_timesteps + 1, row_idx * (max_timesteps + 1) + 1)
    thumb_ax.imshow(Image.open(row.thumbnail_path))
    thumb_ax.axis("off")
    thumb_ax.set_title(
        textwrap.fill(row.title, 18) + f"\n{row.view_count:,}",
        fontsize=7,
    )

    for t in range(max_timesteps):
        subplot_index = row_idx * (max_timesteps + 1) + (t + 2)
        ax = master_fig.add_subplot(len(results_df), max_timesteps + 1, subplot_index, projection="3d")
        if t < n_timesteps:
            plotter.plot_surf(
                preds[t],
                axes=ax,
                views="left",
                cmap="fire",
                norm_percentile=99,
                vmin=0.6,
                alpha_cmap=(0, 0.2),
            )
            trow = time_rows.iloc[t]
            ax.set_title(f"t={t}\n{trow.top_roi}", fontsize=6)
        else:
            ax.set_axis_off()

master_fig.suptitle("Thumbnail study: master contact sheet", fontsize=16)
plt.subplots_adjust(left=0.02, right=0.99, top=0.985, bottom=0.03, wspace=0.05, hspace=0.22)
master_sheet_path = OUT_FOLDER / "thumbnail-study-master-contact-sheet.png"
master_fig.savefig(master_sheet_path, dpi=180, bbox_inches="tight", pad_inches=0.2)
plt.show()

display(timestep_manifest_df)
print(f"Saved: {brain_gallery_path}")
print(f"Saved: {timestep_manifest_path}")
print(f"Saved: {timestep_zip_path}")
print(f"Saved: {master_sheet_path}")


## Metric glossary

Plain-English meaning of the main exported metrics:

- `peak_top_roi_score`: the single strongest ROI score seen at any timestep for a thumbnail.
- `mean_top_roi_score`: for each timestep, take the strongest ROI score; then average those per-timestep top ROI scores across the clip.
- `peak_vertex_response`: the strongest single vertex value seen anywhere in the predicted brain map across all timesteps.
- `mean_vertex_response`: the mean value of all predicted vertex responses across the clip.
- `visual_mean`: average response across the notebook's visual ROI group, averaged over timesteps.
- `auditory_speech_mean`: average response across the notebook's auditory and speech ROI group, averaged over timesteps.
- `language_semantic_mean`: average response across the notebook's language and semantic ROI group, averaged over timesteps.
- `attention_control_mean`: average response across the notebook's attention/control ROI group, averaged over timesteps.
- `motor_somato_mean`: average response across the notebook's motor and somatosensory ROI group, averaged over timesteps.
- `visual_peak`: the maximum timestep-level average of the visual ROI group.
- `attention_control_peak`: the maximum timestep-level average of the attention/control ROI group.
- `peak_top_roi`: the ROI name associated with `peak_top_roi_score`.
- `peak_timestep`: the timestep where `peak_top_roi_score` occurred.

Interpretation note:
- ROI-group metrics depend on the notebook's hand-defined ROI families. They are useful for within-study comparison, not as canonical neuroscience labels.


## Correlation analysis: brain scores vs views

We correlate current YouTube views with a few clip-level brain metrics. Because the view distribution is highly skewed, we use `log10(views)` for the x-axis.


In [ ]:
analysis_df = results_df.copy()
analysis_df["log10_views"] = np.log10(analysis_df["view_count"])

metric_cols = [
    "peak_top_roi_score",
    "mean_top_roi_score",
    "peak_vertex_response",
    "mean_vertex_response",
    "visual_mean",
    "auditory_speech_mean",
    "language_semantic_mean",
    "attention_control_mean",
    "motor_somato_mean",
]

corr_rows = []
for metric in metric_cols:
    pearson_r = analysis_df[["log10_views", metric]].corr(method="pearson").iloc[0, 1]
    spearman_r = analysis_df[["log10_views", metric]].corr(method="spearman").iloc[0, 1]
    slope, intercept = np.polyfit(analysis_df["log10_views"], analysis_df[metric], 1)
    corr_rows.append(
        {
            "metric": metric,
            "pearson_r": float(pearson_r),
            "spearman_r": float(spearman_r),
            "slope": float(slope),
            "intercept": float(intercept),
        }
    )

corr_df = pd.DataFrame(corr_rows).sort_values("spearman_r", ascending=False)
corr_path = OUT_FOLDER / "thumbnail-study-correlations.csv"
corr_df.to_csv(corr_path, index=False)
display(corr_df)
print(f"Saved: {corr_path}")
print("Note: this notebook reports correlation coefficients only. With n=10, treat them as descriptive, not inferential.")

scatter_metrics = [
    "peak_top_roi_score",
    "mean_top_roi_score",
    "visual_mean",
    "attention_control_mean",
]

colors = {"top_views": "tab:red", "low_views": "tab:blue", "fill": "tab:gray"}
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, metric in zip(axes.flat, scatter_metrics):
    for bucket, bucket_df in analysis_df.groupby("selection_bucket"):
        ax.scatter(
            bucket_df["view_count"],
            bucket_df[metric],
            label=bucket,
            color=colors.get(bucket, "tab:gray"),
            s=70,
            alpha=0.85,
        )
    for row in analysis_df.itertuples():
        ax.annotate(row.video_id, (row.view_count, getattr(row, metric)), fontsize=7, alpha=0.8)
    ax.set_xscale("log")
    ax.set_xlabel("views (log scale)")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(alpha=0.2)

handles, labels = axes.flat[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=3)

plt.tight_layout(rect=(0, 0, 1, 0.96))
scatter_path = OUT_FOLDER / "thumbnail-study-correlation-scatterplots.png"
plt.savefig(scatter_path, dpi=180, bbox_inches="tight")
plt.show()

print(f"Saved: {scatter_path}")

focus_metrics = ["mean_top_roi_score", "peak_top_roi_score"]
detail_rows = []
fig, axes = plt.subplots(1, len(focus_metrics), figsize=(14, 5))
if len(focus_metrics) == 1:
    axes = [axes]

for ax, metric in zip(axes, focus_metrics):
    x = analysis_df["log10_views"].to_numpy(dtype=float)
    y = analysis_df[metric].to_numpy(dtype=float)
    pearson_r = float(np.corrcoef(x, y)[0, 1])
    spearman_r = float(analysis_df[["log10_views", metric]].corr(method="spearman").iloc[0, 1])
    slope, intercept = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    y_line = slope * x_line + intercept

    for bucket, bucket_df in analysis_df.groupby("selection_bucket"):
        ax.scatter(
            bucket_df["log10_views"],
            bucket_df[metric],
            s=85,
            alpha=0.9,
            color=colors.get(bucket, "tab:gray"),
            label=bucket,
        )
    for row in analysis_df.itertuples():
        ax.annotate(row.video_id, (row.log10_views, getattr(row, metric)), xytext=(4, 4), textcoords="offset points", fontsize=8, alpha=0.85)
    ax.plot(x_line, y_line, color="black", linewidth=2, alpha=0.8)
    ax.set_xlabel("log10(views)")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(alpha=0.2)
    ax.text(
        0.03,
        0.97,
        f"Pearson r = {pearson_r:.3f}\nSpearman r = {spearman_r:.3f}\nSlope = {slope:.3f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=10,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.85, edgecolor="0.7"),
    )

    for i in range(len(analysis_df)):
        mask = np.ones(len(analysis_df), dtype=bool)
        mask[i] = False
        loo_r = float(np.corrcoef(x[mask], y[mask])[0, 1])
        detail_rows.append(
            {
                "metric": metric,
                "video_id": analysis_df.iloc[i]["video_id"],
                "title": analysis_df.iloc[i]["title"],
                "view_count": int(analysis_df.iloc[i]["view_count"]),
                "metric_value": float(analysis_df.iloc[i][metric]),
                "pearson_r_full": pearson_r,
                "pearson_r_without_point": loo_r,
                "abs_r_change": abs(pearson_r - loo_r),
            }
        )

handles, labels = axes[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc="upper center", ncol=3)
plt.tight_layout(rect=(0, 0, 1, 0.92))
detail_plot_path = OUT_FOLDER / "thumbnail-study-focus-regressions.png"
plt.savefig(detail_plot_path, dpi=180, bbox_inches="tight")
plt.show()

detail_df = pd.DataFrame(detail_rows).sort_values(["metric", "abs_r_change"], ascending=[True, False])
detail_csv_path = OUT_FOLDER / "thumbnail-study-regression-and-influence.csv"
detail_df.to_csv(detail_csv_path, index=False)
top_influence_df = detail_df.groupby("metric", group_keys=False).head(3).reset_index(drop=True)
top_influence_path = OUT_FOLDER / "thumbnail-study-top-influential-points.csv"
top_influence_df.to_csv(top_influence_path, index=False)
display(detail_df)
display(top_influence_df)
print(f"Saved: {detail_plot_path}")
print(f"Saved: {detail_csv_path}")
print(f"Saved: {top_influence_path}")


## Download outputs

This exports the selected-video metadata, per-clip results, per-timestep ROI table, and the generated figures.


In [ ]:
download_paths = [
    OUT_FOLDER / "selected-thumbnail-videos.csv",
    OUT_FOLDER / "thumbnail-study-results.csv",
    OUT_FOLDER / "thumbnail-study-roi-timeseries.csv",
    OUT_FOLDER / "thumbnail-study-correlations.csv",
    OUT_FOLDER / "thumbnail-study-regression-and-influence.csv",
    OUT_FOLDER / "thumbnail-study-top-influential-points.csv",
    OUT_FOLDER / "thumbnail-study-timestep-galleries-manifest.csv",
    OUT_FOLDER / "selected-thumbnails.png",
    OUT_FOLDER / "thumbnail-study-peak-brain-gallery.png",
    OUT_FOLDER / "thumbnail-study-master-contact-sheet.png",
    OUT_FOLDER / "thumbnail-study-correlation-scatterplots.png",
    OUT_FOLDER / "thumbnail-study-focus-regressions.png",
    OUT_FOLDER / "thumbnail-study-timestep-galleries.zip",
]

for path in download_paths:
    print(path)

if IN_COLAB:
    for path in download_paths:
        files.download(str(path))
